# RL-VVC V6 — Action-Conditioned D3QN Reconstruction (Train16 / Val4)

This notebook trains the **new V6 implementation**, not historical V5. It tests whether explicit action-conditioned Q learning and coverage-aware exploration produce better **state-dependent action decisions**, better native Val4 objective and stronger action-value calibration.

**Experiment contract:** locked BDD100K Mini Train16/Val4, 256 coding-order pictures/episode, target Val4=120 kbps; native VVenC + VVdeC parity; frozen source YOLOv5n pseudo-F1. Historical **Test30 is never decoded**. Objective/reward and 25 physical QP actions are preserved for comparability; V6 replaces state encoder, action-value architecture, exploration and replay sampler, and tests an 8-step return. These are coupled design choices, **not** a one-factor ablation against V5.

**Prerequisites:** Kaggle Internet ON, GPU T4 ON, attach `robikscube/driving-video-with-object-tracking`; first run fresh (`RESUME_INPUT=None`). Stop/save automatically after completed video episodes if approaching runtime budget. Do not resume a V5 checkpoint.

**Research caveats:** pseudo-F1 is not ground-truth mAP; same target rate is not actual-rate matched BD-rate; one seed and Val4 are development evidence only. A PASS on unit/preflight does not imply native improvement.

In [ ]:
import sys, os, json, time, zipfile, uuid, shutil, subprocess, traceback, hashlib, csv
from pathlib import Path
from statistics import mean

# 76b4f62feb105125dfd35e135520daaefcb6d017 is replaced before the final GitHub notebook push.
TRAINING_SOURCE_COMMIT='76b4f62feb105125dfd35e135520daaefcb6d017'
SOURCE_BRANCH='experiment/d3qn-v6-action-conditioned-reconstruction'
REPO=Path('/kaggle/working/rl_vvc_d3qn_v6_source')
TOOLS=Path('/kaggle/working/rl_vvc_d3qn_v6_tools')
RUN=Path('/kaggle/working/rl_vvc_d3qn_v6_learning_check')
SAFE=Path('/kaggle/working/D3QN_V6_ACTION_CONDITIONED_MINI16_VAL4_SAFE.zip')
INPUT_ROOT=Path('/kaggle/input')
SEED=20261111
DEVICE='cuda:0'
MAX_HOURS=11.75
STARTED=time.time()
TARGET_RATE=120
MILESTONES=(1024,2048,3072,4096,5120)
CONTROLS=('qpa_anchor','global_minus2','roi_minus2_bg_zero','roi_minus1_bg_plus2')
PROTOCOL='native-masked-d3qn-qp/v6-action-conditioned'
RESUME_INPUT=None  # Optional V6 SAFE .zip or Kaggle automatically-extracted V6 directory; NEVER V5.

assert len(TRAINING_SOURCE_COMMIT)==40 and all(c in '0123456789abcdef' for c in TRAINING_SOURCE_COMMIT), 'Notebook not yet pinned to new V6 training source'

def loadjson(path):return json.loads(Path(path).read_text(encoding='utf-8'))
def atomicjson(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(value,indent=2,allow_nan=False),encoding='utf-8');temp.replace(path)
def live(cmd,*,cwd=None,log_file=None):
    cmd=list(map(str,cmd))
    print('$',' '.join(cmd),flush=True)
    logfile=None
    if log_file:
        path=Path(log_file);path.parent.mkdir(parents=True,exist_ok=True)
        logfile=path.open('w',encoding='utf-8')
    process=subprocess.Popen(cmd,cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end='',flush=True)
            if logfile: logfile.write(line);logfile.flush()
        code=process.wait()
    finally:
        if process.poll() is None:
            process.terminate();process.wait()
        if logfile:logfile.close()
    if code:raise RuntimeError(f'Command failed ({code}); see {log_file}')

def sha(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            digest.update(block)
    return digest.hexdigest()

RUN.mkdir(parents=True,exist_ok=True)
print('V6_SOURCE_COMMIT',TRAINING_SOURCE_COMMIT,'TEST30',False)
if not (REPO/'.git').is_dir():
    live(['git','clone','--depth','1','--branch',SOURCE_BRANCH,
          'https://github.com/youngan2810/RL-VVC.git',REPO])
live(['git','fetch','--no-tags','--depth','1','origin',TRAINING_SOURCE_COMMIT],cwd=REPO)
live(['git','checkout','--detach',TRAINING_SOURCE_COMMIT],cwd=REPO)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==TRAINING_SOURCE_COMMIT
assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=REPO,text=True).strip()
if str(REPO) not in sys.path:sys.path.insert(0,str(REPO))
from scripts.rl_rc_dot.data import VideoCatalog
catalog=VideoCatalog(INPUT_ROOT,mode='mini',seed=SEED)
TRAIN_IDS=list(catalog.selected['train']);VAL_IDS=list(catalog.selected['val'])
assert len(TRAIN_IDS)==16 and len(VAL_IDS)==4 and set(TRAIN_IDS).isdisjoint(VAL_IDS)
catalog.check_available(('train','val'))
for video in TRAIN_IDS+VAL_IDS:catalog.source(video)
print('DATA_LOCK_PASS','TRAIN16',len(TRAIN_IDS),'VAL4',len(VAL_IDS))

## Gate A — toolchain and source identity

May take substantial CPU time for native VVenC/VVdeC compilation. SHA of the newly built encoder is recorded; reproducibility of a *historical* binary is not asserted.

In [ ]:
import torch
assert torch.cuda.is_available(),'Enable Kaggle GPU (Accelerator = T4)'
print('TORCH',torch.__version__,'GPU',torch.cuda.get_device_name(0))
if not (TOOLS/'TOOLCHAIN.json').is_file():
    live([sys.executable,'-m','scripts.rl_rc_dot.setup_kaggle',
          '--repo',REPO,'--source-commit',TRAINING_SOURCE_COMMIT,
          '--output',TOOLS,'--jobs','2','--install-system','--d3qn-only'],
          cwd=REPO,log_file=RUN/'logs'/'setup.log')
toolchain=loadjson(TOOLS/'TOOLCHAIN.json')
assert toolchain['source_commit']==TRAINING_SOURCE_COMMIT
for name in ('encoder','decoder','yolo_repo','weights'):
    assert Path(toolchain[name]).exists(),f'Missing pinned {name}'
assert sha(toolchain['encoder'])==toolchain['encoder_sha256']
assert sha(toolchain['decoder'])==toolchain['decoder_sha256']
print('V6_TOOLCHAIN_READY',toolchain['encoder_sha256'][:16],toolchain['decoder_sha256'][:16])

## Gate B — CPU regressions + real codec preflight (mandatory)

Unit tests check synthetic learning **only**, then real native preflight verifies actual QG action mapping, bitstream and independent decoder parity. Failing this gate must stop training.

In [ ]:
live([sys.executable,'-m','unittest',
    'scripts.rl_rc_dot.test_d3qn_native_v2',
    'scripts.rl_rc_dot.test_d3qn_native_v3',
    'scripts.rl_rc_dot.test_d3qn_native_v4',
    'scripts.rl_rc_dot.test_d3qn_native_v5',
    'scripts.rl_rc_dot.test_d3qn_v6_reconstruction',
    'scripts.rl_rc_dot.test_d3qn_v6_diagnostics','-v'],
    cwd=REPO,log_file=RUN/'logs'/'unit_tests.log')
live([sys.executable,'-c',
     'from scripts.rl_rc_dot.pixels import verify_conversion; print(verify_conversion())'],
     cwd=REPO,log_file=RUN/'logs'/'rgb_parity.log')
GATE=RUN/'native_preflight32'
if not (GATE/'PREFLIGHT.json').is_file():
    if GATE.exists():shutil.rmtree(GATE)
    live([sys.executable,'-m','scripts.rl_rc_dot.preflight',
          '--encoder',toolchain['encoder'],'--decoder',toolchain['decoder'],
          '--output',GATE,'--frames','32','--threads','2'],
          cwd=REPO,log_file=RUN/'logs'/'preflight.log')
preflight=loadjson(GATE/'PREFLIGHT.json')
assert preflight['gate']=='REAL_VVC_STATE_ACTION_AU_DECODE_PASS'
assert preflight['encoder_sha256']==toolchain['encoder_sha256']
print('MANDATORY_V6_PREFLIGHT_PASS')

## Gate C — resumable progress and SAFE ZIP

Resume must be V6 only. Kaggle may expose an uploaded ZIP either as a file or as an extracted directory. A failed/partial training stage is salvaged at the most recent complete 256-picture checkpoint.

In [ ]:
CKPT=RUN/'checkpoints';CKPT.mkdir(parents=True,exist_ok=True)
LATEST=CKPT/'latest_checkpoint.pt';BEST=CKPT/'best_checkpoint.pt'
TRAIN=RUN/'train_history.json';BASE=RUN/'baseline.json';MON=RUN/'monitor_history.json'
PARTIAL=RUN/'monitor_partial.json';MANIFEST=RUN/'learning_manifest.json'
FAILURE=RUN/'last_failure.json'

def import_resume(source):
    source=Path(source)
    if source.is_file():
        with zipfile.ZipFile(source) as z:
            keys=[s for s in z.namelist() if s in ('train_history.json','baseline.json',
                  'monitor_history.json','monitor_partial.json','learning_manifest.json')
                  or s.startswith('checkpoints/') or s.startswith('frame_traces/')]
            for key in keys:
                if '..' in Path(key).parts:raise ValueError('Unsafe zip member')
                dest=RUN/key;dest.parent.mkdir(parents=True,exist_ok=True)
                dest.write_bytes(z.read(key))
    elif source.is_dir():
        for name in ('train_history.json','baseline.json','monitor_history.json','monitor_partial.json','learning_manifest.json'):
            if (source/name).is_file():shutil.copyfile(source/name,RUN/name)
        for sub in ('checkpoints','frame_traces'):
            if (source/sub).is_dir():
                shutil.copytree(source/sub,RUN/sub,dirs_exist_ok=True)
    else:raise FileNotFoundError(f'Resume V6 artifact not found: {source}')

if RESUME_INPUT and not MANIFEST.is_file():import_resume(RESUME_INPUT)
training=loadjson(TRAIN) if TRAIN.is_file() else []
baselines=loadjson(BASE) if BASE.is_file() else []
monitors=loadjson(MON) if MON.is_file() else []
partial=loadjson(PARTIAL) if PARTIAL.is_file() else []
if MANIFEST.is_file():
    state=loadjson(MANIFEST)
    assert state['source_commit']==TRAINING_SOURCE_COMMIT and state['protocol']==PROTOCOL
    assert state['seed']==SEED and not state['test_used']
if training:
    assert len(training)<=20 and all(x['total_steps']==256*(i+1) for i,x in enumerate(training))
    assert LATEST.exists(),'Cannot resume training without last complete V6 checkpoint'
assert all(r['method'] in CONTROLS and r['video'] in VAL_IDS for r in baselines)

def best_point():
    return max(monitors,key=lambda x:(x['mean_objective'],-x['steps'])) if monitors else None

def save_safe(status='partial'):
    for path,data in ((TRAIN,training),(BASE,baselines),(MON,monitors),(PARTIAL,partial)):
        atomicjson(path,data)
    best=best_point()
    atomicjson(MANIFEST,{'schema':'d3qn-v6-action-conditioned-mini16-val4/1',
          'source_commit':TRAINING_SOURCE_COMMIT,'protocol':PROTOCOL,'seed':SEED,
          'train_video_count':16,'validation_video_count':4,'test_used':False,
          'train_steps':training[-1]['total_steps'] if training else 0,
          'completed_episodes':len(training),'validation_milestones':[m['steps'] for m in monitors],
          'baseline_points':len(baselines),'partial_monitor_points':len(partial),
          'best_steps':best['steps'] if best else None,
          'best_objective':best['mean_objective'] if best else None,'status':status})
    temp=SAFE.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED,compresslevel=4) as z:
        for path in [TRAIN,BASE,MON,PARTIAL,MANIFEST,FAILURE,
                     RUN/'v6_diagnostics.json',RUN/'v6_validation_by_video.csv',GATE/'PREFLIGHT.json',LATEST,BEST]:
            if path.is_file():z.write(path,path.relative_to(RUN).as_posix())
        for pat in ('checkpoints/at_*.pt','frame_traces/*.jsonl.gz','logs/*.log'):
            for file in RUN.glob(pat):z.write(file,file.relative_to(RUN).as_posix())
        z.writestr('TOOLCHAIN_FINGERPRINT.json',json.dumps(toolchain,indent=2))
    temp.replace(SAFE)
    print('V6_SAFE_EXPORTED',status,SAFE.stat().st_size,flush=True)

def runner_cmd(phase,output,*,checkpoint=None,steps=1024,val_index=0,methods=None):
    cmd=[sys.executable,'-m','scripts.rl_rc_dot.d3qn_v6_native_runner',
        '--phase',phase,'--input-root',INPUT_ROOT,
        '--encoder',toolchain['encoder'],'--decoder',toolchain['decoder'],
        '--yolo-repo',toolchain['yolo_repo'],'--weights',toolchain['weights'],
        '--output',output,'--device',DEVICE,'--seed',SEED,
        '--n-step',8,'--action-mix',.45,'--boundary-mix',.10,
        '--target-tau',.005,'--learning-rate',.00005,
        '--epsilon-floor',.20,'--epsilon-decay-steps',10240,
        '--encoder-threads',2,'--train-steps',steps,
        '--val-videos',1,'--val-start',val_index,'--val-rates',TARGET_RATE,
        '--max-hours',MAX_HOURS,'--session-start-utc',STARTED]
    if checkpoint:cmd+=['--checkpoint-in',checkpoint]
    if methods:cmd+=['--val-methods',methods]
    return cmd

def allowed_new_stage():
    return time.time()-STARTED < MAX_HOURS*3600-1200

def capture_traces(folder,label):
    target=RUN/'frame_traces';target.mkdir(exist_ok=True)
    for path in (folder/'validation').rglob('trajectory.jsonl.gz'):
        name=path.parent.parent.name  # policy-specific outer folder
        filename=f'{label}_{path.parent.name}_{name}_{len(list(target.glob(label+"*")))}.jsonl.gz'
        shutil.copyfile(path,target/filename)

def attempt(stage,phase,*,checkpoint=None,steps=1024,val_index=0,methods=None):
    out=RUN/'attempts'/(stage+'_'+uuid.uuid4().hex[:8]);out.parent.mkdir(parents=True,exist_ok=True)
    log=RUN/'logs'/(out.name+'.log')
    try:
        live(runner_cmd(phase,out,checkpoint=checkpoint,steps=steps,val_index=val_index,
                        methods=methods),cwd=REPO,log_file=log)
    except Exception:
        # Salvage only complete episodes (the V6 agent saves atomically at 256-picture boundaries).
        if phase=='train' and (out/'checkpoint.pt').is_file() and (out/'training.json').is_file():
            rows=loadjson(out/'training.json')
            completed=training[-1]['total_steps'] if training else 0
            for r in rows:
                if r['total_steps'] == completed+256:
                    training.append(r);completed+=256
            shutil.copyfile(out/'checkpoint.pt',LATEST)
            save_safe('training_interrupted_checkpoint_salvaged')
        raise
    return out

save_safe('restored' if training else 'initial')
print('RESUME',len(training),'episodes',len(monitors),'milestones')

## Gate D — fixed baselines (once per Val4 video)

Verify fixed action 2, fixed action 9, original QPA and global ΔQP −2 on **actual reconstructed video**. Each case includes independent VVdeC parity.

In [ ]:
stop=False
try:
    for i,vid in enumerate(VAL_IDS):
        existing=[x for x in baselines if x['video']==vid]
        if existing:
            assert {x['method'] for x in existing}==set(CONTROLS)
            print('V6_BASELINE_REUSED',vid);continue
        if not allowed_new_stage():stop=True;print('V6_BASELINE_BUDGET_STOP',vid);break
        folder=attempt('baseline_'+str(i),'probe',val_index=i,methods=','.join(CONTROLS))
        rows=loadjson(folder/'validation.json')
        assert len(rows)==len(CONTROLS) and all(x['video']==vid and x['decoder_parity'] for x in rows)
        capture_traces(folder,'baseline_'+str(i))
        baselines.extend(rows)
        save_safe('baseline_checkpoint')
except Exception as exc:
    atomicjson(FAILURE,{'stage':'baseline','error':str(exc),'traceback':traceback.format_exc()[-9000:]})
    save_safe('failed_baseline');raise
finally:
    save_safe('baseline_ready' if len(baselines)==16 else 'baseline_partial')
print('V6_BASELINES',len(baselines),'/16')
if len(baselines)==16:
    for name in CONTROLS:
        rows=[x['summary'] for x in baselines if x['method']==name]
        print(name,'J',round(mean(x['objective'] for x in rows),6),
              'F1',round(mean(x['mean_frame_f1'] for x in rows),6),
              'kbps',round(mean(x['actual_kbps'] for x in rows),4))

## Gate E — staged V6 training + five independent Val4 checkpoints

This is the only cell that trains V6. The notebook runs exactly **20×256 native episodes for the initial learning check**, with validation every 1,024 transitions. No convergence is guaranteed at 5,120 steps; extend in a separately named protocol if necessary. Historical V5 checkpoints **cannot** be resumed.

In [ ]:
def score_monitor(rows,steps):
    assert len(rows)==4 and {r['video'] for r in rows}==set(VAL_IDS)
    assert all(r['method']=='trained_d3qn_v6' and r['rate_kbps']==TARGET_RATE
               and r['decoder_parity'] for r in rows)
    hist=[sum(r['summary']['actions'][i] for r in rows) for i in range(25)]
    assert sum(hist)==1024
    return {'steps':steps,'results':rows,'mean_objective':mean(r['summary']['objective'] for r in rows),
            'mean_f1':mean(r['summary']['mean_frame_f1'] for r in rows),
            'mean_actual_kbps':mean(r['summary']['actual_kbps'] for r in rows),
            'action_histogram':hist,'dominant_action':int(max(range(25),key=lambda i:hist[i])),
            'dominant_fraction':max(hist)/1024}
try:
    if len(baselines)!=16:print('V6_TRAIN_SKIPPED: baselines incomplete; resume with V6 SAFE ZIP')
    else:
        for milestone in MILESTONES:
            if any(x['steps']==milestone for x in monitors):
                print('V6_MILESTONE_REUSED',milestone);continue
            completed=training[-1]['total_steps'] if training else 0
            assert completed<=milestone,'Unvalidated training milestone was skipped'
            at=CKPT/f'at_{milestone}.pt'
            if completed<milestone:
                if not allowed_new_stage():stop=True;print('V6_TRAIN_BUDGET_STOP');break
                folder=attempt('train_to_'+str(milestone),'train',
                    checkpoint=LATEST if completed else None,steps=milestone-completed)
                rows=loadjson(folder/'training.json')
                assert rows and rows[0]['total_steps']==completed+256
                for r in rows:
                    assert r['total_steps']==(training[-1]['total_steps'] if training else 0)+256
                    training.append(r)
                shutil.copyfile(folder/'checkpoint.pt',LATEST)
                save_safe('training_checkpoint')
            if training[-1]['total_steps']<milestone:
                stop=True;print('V6_TRAIN_PARTIAL_STOP',milestone);break
            if not at.is_file():shutil.copyfile(LATEST,at)
            for i,vid in enumerate(VAL_IDS):
                if any(x['steps']==milestone and x['video']==vid for x in partial):
                    print('V6_MONITOR_REUSED',milestone,vid);continue
                if not allowed_new_stage():stop=True;print('V6_MONITOR_BUDGET_STOP');break
                folder=attempt('monitor_'+str(milestone)+'_'+str(i),'validate',
                    checkpoint=at,val_index=i,methods='trained_d3qn_v6')
                rows=loadjson(folder/'validation.json')
                assert len(rows)==1 and rows[0]['video']==vid and rows[0]['decoder_parity']
                capture_traces(folder,f'monitor_{milestone}_{i}')
                partial.append({'steps':milestone,**rows[0]})
                save_safe('monitor_video_checkpoint')
            if stop:break
            complete=[{k:v for k,v in x.items() if k!='steps'} for x in partial if x['steps']==milestone]
            point=score_monitor(complete,milestone)
            old=best_point()
            if old is None or point['mean_objective']>old['mean_objective']:
                shutil.copyfile(at,BEST)
            monitors.append(point)
            partial[:]=[x for x in partial if x['steps']!=milestone]
            save_safe('milestone_complete')
            print('V6_VAL4_PASS',milestone,'J',round(point['mean_objective'],6),
                'dominant',point['dominant_action'],'fraction',round(point['dominant_fraction'],4),
                'best_step',best_point()['steps'])
except Exception as exc:
    atomicjson(FAILURE,{'stage':'training_or_monitor','error':str(exc),
                        'traceback':traceback.format_exc()[-12000:]})
    save_safe('error');raise
finally:
    save_safe('complete' if len(training)==20 and len(monitors)==5 else 'partial')
print('V6_PROGRESS',len(training),'episodes',len(monitors),'milestones')

## Gate F — result diagnostics and handoff

Interpret Val4 objective, actual bitrate, action distribution and per-video deltas together. Greedy action diversity alone is not a successful objective. Print comparison against historical **V5 best J=0.649158** and **Fixed action 2 J=0.658950**; these historical development values are NOT rate-matched BD-rate.

In [ ]:
import numpy as np
summary=[]
for row in monitors:
    summary.append({'steps':row['steps'],'objective_J':row['mean_objective'],
                    'pseudo_F1':row['mean_f1'],'actual_kbps':row['mean_actual_kbps'],
                    'dominant_action':row['dominant_action'],
                    'dominant_fraction':row['dominant_fraction']})
print('V6_VALIDATION_SUMMARY',json.dumps(summary,indent=2))
for name in ('roi_minus2_bg_zero','roi_minus1_bg_plus2'):
    vals=[r['summary'] for r in baselines if r['method']==name]
    if len(vals)==4:print('V6_NATIVE_FIXED',name,'J',mean(x['objective'] for x in vals))
if monitors:
    best=best_point()
    delta=best['mean_objective']-0.658950367
    print('V6_BEST',best['steps'],'J',best['mean_objective'],'vs_historical_fixed2_delta',delta)
    print('V6_NOTE', 'NO CLAIM OF IMPROVEMENT WITHOUT MATCHED NATIVE EVALUATION AND MULTIPLE SEEDS')
    print('V6_ACTION_HIST',best['action_histogram'])
    with (RUN/'v6_validation_by_video.csv').open('w',newline='') as file:
        writer=csv.writer(file)
        writer.writerow(['checkpoint_steps','video','objective_J','pseudo_F1','actual_kbps'])
        for m in monitors:
            for r in m['results']:
                s=r['summary'];writer.writerow([m['steps'],r['video'],s['objective'],
                                               s['mean_frame_f1'],s['actual_kbps']])
if training:
    atomicjson(RUN/'v6_diagnostics.json',{
        'source_commit':TRAINING_SOURCE_COMMIT,
        'train_action_rate_counts':training[-1].get('action_rate_counts'),
        'validation':summary,
        'no_test30':True,
        'caveat':'Action coverage and synthetic tests do not prove learned adaptive quality'})
save_safe('complete' if len(training)==20 and len(monitors)==5 else 'partial')
from IPython.display import FileLink,display
os.chdir('/kaggle/working')
print('FINAL_SAFE_ZIP',SAFE)
display(FileLink(SAFE.name))

## Optional separate native on-policy diagnostic (not training)

The new repository includes `scripts.rl_rc_dot.d3qn_v6_native_pair_probe`, supporting **frozen V5 or V6 greedy prefix**; two independent encodes per step, action 2 vs 9 at the same native state, then frozen-policy suffix. Use on Train16 only. If the encoder binary SHA differs from the frozen checkpoint's, first reproduce a trusted baseline; `--allow-encoder-mismatch` records, but does not establish, comparability. Do not interpret the paired returns as ground-truth Q* or use future outcomes as encoder observations.

The optional command is deliberately **not executed** automatically: it needs a frozen checkpoint and additional expensive native encodes. No Kaggle jobs are launched by the assistant.